# 01 – Data preparation

Download 5-minute SPY data, keep regular U.S. trading hours, and build the daily realized-volatility dataset.

## Imports and API setup

In [1]:
import time
from getpass import getpass

import numpy as np
import pandas as pd
import requests

In [2]:
API_KEY = getpass("Massive API key: ")

query_params = {
    "adjusted": "true",
    "sort": "asc",
    "limit": 50000,
    "apiKey": API_KEY,
}

Massive API key:  ········


## Helper functions

In [3]:
def fetch_5min_data(start_date, end_date):
    url = f"https://api.massive.com/v2/aggs/ticker/SPY/range/5/minute/{start_date}/{end_date}"
    response = requests.get(url, params=query_params)
    response.raise_for_status()
    return pd.DataFrame(response.json().get("results", []))


def calculate_realized_variance(day):
    day = day.copy()
    day["log_return"] = np.log(day["c"] / day["c"].shift(1))

    # The first 5-minute return uses the opening price.
    first_return = np.log(day["c"].iloc[0] / day["o"].iloc[0])

    return first_return**2 + (day["log_return"].iloc[1:] ** 2).sum()

## Check with one trading day

In [4]:
sample = fetch_5min_data("2026-09-30", "2026-09-30")

sample["datetime"] = pd.to_datetime(
    sample["t"], unit="ms", utc=True
).dt.tz_convert("America/New_York")

sample_regular = sample[
    (sample["datetime"].dt.time >= pd.to_datetime("09:30").time())
    & (sample["datetime"].dt.time < pd.to_datetime("16:00").time())
].copy()

print("5-minute observations:", len(sample_regular))
print("Realized variance:", calculate_realized_variance(sample_regular))

5-minute observations: 78
Realized variance: 2.781711721654064e-05


## Download the full sample

In [5]:
months = pd.period_range("2024-10", "2026-09", freq="M")
all_data = []

for month in months:
    start_date = month.start_time.strftime("%Y-%m-%d")
    end_date = month.end_time.strftime("%Y-%m-%d")

    print(f"Downloading {start_date} to {end_date}...")
    all_data.append(fetch_5min_data(start_date, end_date))

    # Basic-plan rate limit :'D
    time.sleep(13)

In [6]:
raw = pd.concat(all_data, ignore_index=True)

raw["datetime"] = pd.to_datetime(
    raw["t"], unit="ms", utc=True
).dt.tz_convert("America/New_York")

print("Rows:", len(raw))
print("First observation:", raw["datetime"].min())
print("Last observation:", raw["datetime"].max())
print("Duplicate timestamps:", raw["t"].duplicated().sum())

Rows: 94550
First observation: 2024-10-04 04:00:00-04:00
Last observation: 2026-09-30 19:55:00-04:00
Duplicate timestamps: 0


In [7]:
raw.to_csv(
    "../data/raw/spy_5min_2024-10-03_2026-09-30.csv",
    index=False,
)

## Build the daily realized-volatility series

In [8]:
regular = raw[
    (raw["datetime"].dt.time >= pd.to_datetime("09:30").time())
    & (raw["datetime"].dt.time < pd.to_datetime("16:00").time())
].copy()

regular["date"] = regular["datetime"].dt.date

daily_counts = regular.groupby("date").size()
# Number of trading days by number of 5-minute observations
daily_counts.value_counts().sort_index()

43      5
78    493
Name: count, dtype: int64

In [9]:
# Early-close days have fewer than 78 five-minute intervals.
short_days = daily_counts[daily_counts != 78]
short_days

date
2024-11-29    43
2024-12-24    43
2025-07-03    43
2025-11-28    43
2025-12-24    43
dtype: int64

In [10]:
full_days = daily_counts[daily_counts == 78].index
regular = regular[regular["date"].isin(full_days)].copy()

daily_rv = (
    regular.groupby("date")
    .apply(calculate_realized_variance)
    .rename("realized_variance")
    .reset_index()
)

daily_rv["realized_volatility"] = np.sqrt(daily_rv["realized_variance"])
daily_rv["realized_volatility_percent"] = daily_rv["realized_volatility"] * 100
daily_rv["log_realized_variance"] = np.log(daily_rv["realized_variance"])

daily_rv.head()

,date,realized_variance,realized_volatility,realized_volatility_percent,log_realized_variance
0,2024-10-04,0.000050,0.007089,0.708885,-9.898466
1,2024-10-07,0.000023,0.004801,0.480091,-10.677898
2,2024-10-08,0.000016,0.004034,0.403412,-11.025935
3,2024-10-09,0.000015,0.003849,0.384887,-11.119952
4,2024-10-10,0.000017,0.004162,0.416172,-10.963652


In [11]:
print("Trading days:", len(daily_rv))
print("Missing values:", daily_rv.isna().sum().sum())
print("Finite log-RV:", np.isfinite(daily_rv["log_realized_variance"]).all())

Trading days: 493
Missing values: 0
Finite log-RV: True


In [12]:
daily_rv.to_csv(
    "../data/spy_daily_realized_volatility.csv",
    index=False,
)